# Online Retail Unsupervised EDA

## Problem Statement
This notebook performs complete EDA on raw online retail transactions before unsupervised learning.

## What this notebook covers
- transaction cleaning
- cancellations
- missing values
- invalid rows
- transaction-level EDA
- customer-level aggregation
- RFM
- outlier analysis
- scaling prep
- PCA / t-SNE prep
- cleaned CSV save

## Telugu
Unsupervised model run cheyyadam mundu raw transaction data ni clean chesi, customer-level features create chesi segmentation ki ready cheyyali.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid")

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

## Folder structure expected

```text
eda-online-retail-unsupervised/
├── data/
│   ├── raw/
│   │   └── online_retail.xlsx
│   └── processed/
├── notebooks/
│   └── online_retail_unsupervised_eda.ipynb
├── reports/
│   ├── figures/
│   └── insights.md
└── README.md
```

In [ ]:
df = pd.read_excel("../data/raw/online_retail.xlsx")

print("Shape:", df.shape)
display(df.head())

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print(df.columns.tolist())

In [ ]:
print(df.info())
print("\n")
display(df.describe(include="all").T)

In [ ]:
if "invoicedate" in df.columns:
    df = df.rename(columns={"invoicedate": "invoice_date"})

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

print(df["invoice_date"].min(), df["invoice_date"].max())

## Why parse invoice date?
Recency and time-based aggregation kosam datetime format compulsory.

In [ ]:
missing_report = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_pct", ascending=False)
)

display(missing_report)
missing_report.to_csv("../data/processed/retail_missing_report.csv")

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

# Transaction Cleaning
Retail transaction data lo cancellations, negative quantities, zero price rows untayi.  
Customer segmentation ki valid purchase rows kavali.

In [ ]:
df["is_cancellation"] = df["invoiceno"].astype(str).str.startswith("C")

display(df["is_cancellation"].value_counts())

In [ ]:
print("Rows with quantity <= 0:", (df["quantity"] <= 0).sum())
print("Rows with unitprice <= 0:", (df["unitprice"] <= 0).sum())
print("Rows with missing customerid:", df["customerid"].isna().sum())

In [ ]:
df_txn = df.copy()

df_txn = df_txn[~df_txn["is_cancellation"]]
df_txn = df_txn[df_txn["quantity"] > 0]
df_txn = df_txn[df_txn["unitprice"] > 0]
df_txn = df_txn[df_txn["customerid"].notna()]

print("Clean transaction shape:", df_txn.shape)
display(df_txn.head())

## Why remove these rows?
Customer segmentation కి:
- cancellations vadakudadhu
- negative quantity vadakudadhu
- zero price useful kadu
- missing customer id unte customer-level aggregation possible kadu

In [ ]:
df_txn["customerid"] = df_txn["customerid"].astype(str).str.replace(".0", "", regex=False)

In [ ]:
df_txn["sales_amount"] = df_txn["quantity"] * df_txn["unitprice"]

display(df_txn[["quantity", "unitprice", "sales_amount"]].head())

In [ ]:
txn_num_cols = df_txn.select_dtypes(include=[np.number]).columns.tolist()

numeric_summary = []

for col in txn_num_cols:
    arr = df_txn[col].dropna().to_numpy()
    if arr.size == 0:
        continue

    numeric_summary.append({
        "column": col,
        "count": arr.size,
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
        "var": float(np.var(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "p01": float(np.percentile(arr, 1)),
        "p05": float(np.percentile(arr, 5)),
        "p25": float(np.percentile(arr, 25)),
        "p50": float(np.percentile(arr, 50)),
        "p75": float(np.percentile(arr, 75)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99))
    })

retail_numeric_summary = pd.DataFrame(numeric_summary).sort_values("column")
display(retail_numeric_summary)
retail_numeric_summary.to_csv("../data/processed/retail_numeric_summary.csv", index=False)

In [ ]:
retail_cat_summary = []

for col in cat_cols:
    if col not in df_txn.columns:
        continue
    mode_series = df_txn[col].mode(dropna=True)
    retail_cat_summary.append({
        "column": col,
        "n_unique": int(df_txn[col].nunique(dropna=True)),
        "most_frequent": mode_series.iloc[0] if len(mode_series) > 0 else np.nan,
        "missing_count": int(df_txn[col].isna().sum()),
        "missing_pct": round(df_txn[col].isna().mean() * 100, 2)
    })

retail_cat_summary_df = pd.DataFrame(retail_cat_summary).sort_values("n_unique", ascending=False)
display(retail_cat_summary_df)
retail_cat_summary_df.to_csv("../data/processed/retail_categorical_summary.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_txn["quantity"], bins=50, kde=True)
plt.xlim(0, df_txn["quantity"].quantile(0.99))
plt.title("Quantity Distribution (trimmed to 99th percentile view)")
plt.tight_layout()
plt.savefig("../reports/figures/quantity_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_txn["unitprice"], bins=50, kde=True)
plt.xlim(0, df_txn["unitprice"].quantile(0.99))
plt.title("UnitPrice Distribution (trimmed to 99th percentile view)")
plt.tight_layout()
plt.savefig("../reports/figures/unitprice_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_txn["sales_amount"], bins=50, kde=True)
plt.xlim(0, df_txn["sales_amount"].quantile(0.99))
plt.title("Sales Amount Distribution (trimmed to 99th percentile view)")
plt.tight_layout()
plt.savefig("../reports/figures/sales_amount_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## Why trimmed views?
Retail values long-tail ga untayi. 99th percentile varaku view better readability isthundi.

In [ ]:
country_summary = (
    df_txn.groupby("country", as_index=False)
          .agg(
              transaction_count=("invoiceno", "count"),
              unique_customers=("customerid", "nunique"),
              total_sales=("sales_amount", "sum")
          )
          .sort_values("transaction_count", ascending=False)
)

display(country_summary.head(20))
country_summary.to_csv("../data/processed/retail_country_summary.csv", index=False)

In [ ]:
top_countries = country_summary.head(15)

plt.figure(figsize=(12, 5))
sns.barplot(data=top_countries, x="country", y="transaction_count")
plt.xticks(rotation=45, ha="right")
plt.title("Top Countries by Transaction Count")
plt.tight_layout()
plt.savefig("../reports/figures/country_transaction_counts.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
outlier_rows = []

for col in ["quantity", "unitprice", "sales_amount"]:
    arr = df_txn[col].dropna().to_numpy()
    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = int(((arr < lower) | (arr > upper)).sum())
    outlier_pct = round((outlier_count / len(arr)) * 100, 2)

    outlier_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_pct
    })

retail_outlier_report = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
display(retail_outlier_report)
retail_outlier_report.to_csv("../data/processed/retail_outlier_report.csv", index=False)

## Why outlier report?
Transaction values lo extreme points common.  
Aggregation mundu identify cheyyali.

In [ ]:
reference_date = df_txn["invoice_date"].max() + pd.Timedelta(days=1)
print("Reference date:", reference_date)

In [ ]:
customer_features = (
    df_txn.groupby("customerid")
          .agg(
              recency_days=("invoice_date", lambda x: (reference_date - x.max()).days),
              frequency_orders=("invoiceno", "nunique"),
              monetary_total=("sales_amount", "sum"),
              total_quantity=("quantity", "sum"),
              unique_products=("stockcode", "nunique"),
              unique_invoices=("invoiceno", "nunique"),
              avg_unit_price=("unitprice", "mean"),
              avg_order_value=("sales_amount", "mean"),
              country_main=("country", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
          )
          .reset_index()
)

display(customer_features.head())
print("Customer feature shape:", customer_features.shape)

## Why customer-level aggregation?
Raw transactions ni direct cluster cheyyamu.  
Customer segmentation ki customer-level features kavali.

In [ ]:
customer_features.to_csv("../data/processed/retail_customer_features.csv", index=False)
print("Saved: ../data/processed/retail_customer_features.csv")

In [ ]:
df_txn.to_csv("../data/processed/retail_transaction_cleaned.csv", index=False)
print("Saved: ../data/processed/retail_transaction_cleaned.csv")

In [ ]:
rfm_cols = ["recency_days", "frequency_orders", "monetary_total"]

rfm_summary = customer_features[rfm_cols].describe().T
display(rfm_summary)
rfm_summary.to_csv("../data/processed/retail_rfm_summary.csv")

In [ ]:
customer_features[rfm_cols].hist(figsize=(14, 4), bins=30)
plt.suptitle("RFM Distributions", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/rfm_distributions.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
cust_num_cols = customer_features.select_dtypes(include=[np.number]).columns.tolist()

cust_summary = []

for col in cust_num_cols:
    arr = customer_features[col].dropna().to_numpy()
    cust_summary.append({
        "column": col,
        "count": arr.size,
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
        "var": float(np.var(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "p01": float(np.percentile(arr, 1)),
        "p05": float(np.percentile(arr, 5)),
        "p25": float(np.percentile(arr, 25)),
        "p50": float(np.percentile(arr, 50)),
        "p75": float(np.percentile(arr, 75)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99))
    })

cust_summary_df = pd.DataFrame(cust_summary).sort_values("column")
display(cust_summary_df)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(customer_features[cust_num_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Customer Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
cluster_cols = [
    "recency_days",
    "frequency_orders",
    "monetary_total",
    "total_quantity",
    "unique_products",
    "avg_order_value"
]

cluster_df = customer_features[["customerid"] + cluster_cols].copy()
display(cluster_df.head())

In [ ]:
cluster_df_transformed = cluster_df.copy()

for col in ["frequency_orders", "monetary_total", "total_quantity", "unique_products", "avg_order_value"]:
    cluster_df_transformed[col] = np.log1p(cluster_df_transformed[col])

display(cluster_df_transformed.head())

## Why log transform here?
Customer behavior features heavy skew untayi.  
Distance-based clustering stable ga undadaniki log transform useful.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df_transformed[cluster_cols])

print("Scaled shape:", X_scaled.shape)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["pc1", "pc2"])
pca_df["customerid"] = cluster_df["customerid"].values
display(pca_df.head())

print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
fig = px.scatter(
    pca_df,
    x="pc1",
    y="pc2",
    hover_data=["customerid"],
    title="PCA Projection of Customers"
)
fig.write_html("../reports/figures/pca_customer_clusters.html")
fig.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X_scaled)

tsne_df = pd.DataFrame(X_tsne, columns=["tsne1", "tsne2"])
tsne_df["customerid"] = cluster_df["customerid"].values
display(tsne_df.head())

In [ ]:
fig = px.scatter(
    tsne_df,
    x="tsne1",
    y="tsne2",
    hover_data=["customerid"],
    title="t-SNE Projection of Customers"
)
fig.write_html("../reports/figures/tsne_customer_clusters.html")
fig.show()

In [ ]:
insights = '''
# Online Retail Unsupervised EDA Insights

1. Raw transaction data contains cancellations, non-positive quantities/prices, and missing customer IDs that must be cleaned before customer segmentation.
2. Transaction-level quantity, unit price, and sales amount are strongly skewed and contain extreme values.
3. Customer-level aggregation is necessary before clustering; raw transaction rows are not the right direct input for customer segmentation.
4. RFM features (Recency, Frequency, Monetary) form the core of customer segmentation.
5. Additional customer behavior features such as total quantity, unique products, and average order value add richer signal for clustering.
6. Log transformation and feature scaling are important before distance-based unsupervised algorithms.
7. Cleaned transaction-level and customer-level CSV files were saved for downstream clustering.
8. PCA and t-SNE projections provide an initial view of customer structure before applying clustering algorithms.
'''

with open("../reports/insights.md", "w", encoding="utf-8") as f:
    f.write(insights)

print(insights)